In [5]:
import tensorflow as tf
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import os

# Check the current version of tensorflow
print(f"Tensorflow Version: {tf.__version__}")
# Check if GPU is active or not
print("GPU is", "available" if tf.config.list_physical_devices('GPU') else "NOt available")

Tensorflow Version: 2.20.0
GPU is available


In [6]:
# Cloning the dataset mirror of PlantVillage-Dataset from github
print("Cloning dataset from GitHub...")
!git clone https://github.com/spMohanty/PlantVillage-Dataset.git

# Verify the data path
base_dir = '/content/PlantVillage-Dataset/raw/color'
classes = os.listdir(base_dir)

print(f"Success! Found {len(classes)} different plant disease categories.")

Cloning dataset from GitHub...
Cloning into 'PlantVillage-Dataset'...
remote: Enumerating objects: 163264, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 163264 (delta 16), reused 25 (delta 9), pack-reused 163229 (from 1)
Receiving objects: 100% (163264/163264), 2.00 GiB | 17.49 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Updating files: 100% (182404/182404), done.
Success! Found 38 different plant disease categories.


In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("Initializing Advanced Data Pipeline (Augmentation)...")

# 1. Training Generator (with Augmentation to prevent memorization)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,      # Rotate randomly up to 20 degrees
    width_shift_range=0.2,  # Shift left/right
    height_shift_range=0.2, # Shift up/down
    zoom_range=0.2,         # Zoom in/out
    horizontal_flip=True,   # Flip like a mirror
    fill_mode='nearest',    # Fill missing pixels seamlessly
    validation_split=0.2
)

# 2. Validation Generator (Strictly NO augmentation, just math normalization)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# 3. Extract & Transform Data
print("Loading Training Pipeline:")
train_data = train_datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

print("\nLoading Validation Pipeline:")
val_data = val_datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Initializing Advanced Data Pipeline (Augmentation)...
Loading Training Pipeline:
Found 43456 images belonging to 38 classes.

Loading Validation Pipeline:
Found 10849 images belonging to 38 classes.


In [8]:
from tensorflow._api.v2.config import optimizer
# Architecture
from tensorflow.keras import layers, models, Model
from tensorflow.keras.applications import MobileNetV2

# Load the base model
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False,
                         weights='imagenet')
# Freeze the base
base_model.trainable = False

# Create a unique "Head" for our specific classes
inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)

# GlobalAveragePooling reduces 7x7x1280 to a 1x1280 vector (saves memory)
x = layers.GlobalAveragePooling2D()(x)

# Dropout prevents "memorization" (overfitting) by randomly turning off 20% of neurons
x = layers.Dropout(0.2)(x)

# Output layer : softmax
num_classes = len(train_data.class_indices)
outputs = layers.Dense(num_classes, activation='softmax')(x)

# Finalize the model
model = Model(inputs, outputs)

# Compile with optimizer
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 38)             │        48,678 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,306,662 (8.80 MB)

 Trainable params: 48,678 (190.15 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print("Preparing for Fine-Tuning...")

# 1. Unfreeze the base model
base_model.trainable = True

# 2. Re-freeze the bottom layers
# MobileNetV2 has ~154 layers total. We leave the last 24 layers to learn the leaves.
fine_tune_at = 130

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# 3. Re-compile with a micro-learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint('plant_model_finetuned.keras', monitor='val_accuracy', save_best_only=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=1, min_lr=1e-7, verbose=1)

print("Starting training... ")

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5,
    callbacks=[early_stop, checkpoint, reduce_lr]
)

Preparing for Fine-Tuning...
Starting training... 
Epoch 1/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 600s 423ms/step - accuracy: 0.6131 - loss: 1.5774 - val_accuracy: 0.8313 - val_loss: 0.6381 - learning_rate: 1.0000e-05
Epoch 2/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 563s 415ms/step - accuracy: 0.8585 - loss: 0.5398 - val_accuracy: 0.9110 - val_loss: 0.3048 - learning_rate: 1.0000e-05
Epoch 3/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 551s 406ms/step - accuracy: 0.9034 - loss: 0.3432 - val_accuracy: 0.9341 - val_loss: 0.2114 - learning_rate: 1.0000e-05
Epoch 4/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 541s 399ms/step - accuracy: 0.9241 - loss: 0.2592 - val_accuracy: 0.9478 - val_loss: 0.1686 - learning_rate: 1.0000e-05
Epoch 5/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 526s 387ms/step - accuracy: 0.9369 - loss: 0.2105 - val_accuracy: 0.9544 - val_loss: 0.1401 - learning_rate: 1.0000e-05


In [11]:
print("Continuing Fine-Tuning for 3 more rounds...")

# We use the same 'history_fine' variable to append the new data
history_continued = model.fit(
    train_data,
    validation_data=val_data,
    epochs=3,
    callbacks=[early_stop, checkpoint, reduce_lr]
)

Continuing Fine-Tuning for 10 more rounds...
Epoch 1/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 555s 408ms/step - accuracy: 0.9464 - loss: 0.1796 - val_accuracy: 0.9580 - val_loss: 0.1300 - learning_rate: 1.0000e-05
Epoch 2/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 566s 417ms/step - accuracy: 0.9519 - loss: 0.1609 - val_accuracy: 0.9607 - val_loss: 0.1179 - learning_rate: 1.0000e-05
Epoch 3/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 555s 408ms/step - accuracy: 0.9567 - loss: 0.1417 - val_accuracy: 0.9642 - val_loss: 0.1085 - learning_rate: 1.0000e-05
Epoch 4/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 543s 399ms/step - accuracy: 0.9614 - loss: 0.1284 - val_accuracy: 0.9638 - val_loss: 0.1044 - learning_rate: 1.0000e-05
Epoch 5/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 554s 408ms/step - accuracy: 0.9642 - loss: 0.1173 - val_accuracy: 0.9690 - val_loss: 0.0893 - learning_rate: 1.0000e-05


In [12]:
from google.colab import files
model.save('plant_doctor.keras')
files.download('plant_doctor.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
classes

['Apple___healthy',
 'Tomato___Spider_mites Two-spotted_spider_mite',
 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
 'Potato___Late_blight',
 'Tomato___Target_Spot',
 'Grape___Black_rot',
 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
 'Tomato___Leaf_Mold',
 'Tomato___Bacterial_spot',
 'Pepper,_bell___healthy',
 'Tomato___healthy',
 'Blueberry___healthy',
 'Peach___Bacterial_spot',
 'Apple___Black_rot',
 'Corn_(maize)___healthy',
 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
 'Strawberry___Leaf_scorch',
 'Potato___healthy',
 'Apple___Apple_scab',
 'Grape___healthy',
 'Cherry_(including_sour)___Powdery_mildew',
 'Peach___healthy',
 'Potato___Early_blight',
 'Cherry_(including_sour)___healthy',
 'Pepper,_bell___Bacterial_spot',
 'Soybean___healthy',
 'Tomato___Tomato_mosaic_virus',
 'Strawberry___healthy',
 'Apple___Cedar_apple_rust',
 'Raspberry___healthy',
 'Tomato___Late_blight',
 'Orange___Haunglongbing_(Citrus_greening)',
 'Grape___Esca_(Black_Measles)',
 'Corn_(maize)___North